This notebook cleans and preprocesses the original LendingClub dataset, removes post-origination leakage, engineers features, applies filtering, and exports a clean dataset for EDA and modeling.

In [11]:
import numpy as np
import pandas as pd
import os

In [3]:
#Load in raw data

df_raw = pd.read_csv("../data/raw_data.csv", low_memory=False)
print(f"Shape: {df_raw.shape}")

Shape: (2260668, 145)


In [4]:
#Remove ambiguous states, create default flag

df_base = df_raw.copy()

#Ambiguous states: Loan not fully resolved
ambiguous = [
    'Current',
    'In Grace Period',
    'Late (16-30 days)',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off'
]

df_base = df_base[~df_base['loan_status'].isin(ambiguous)]
df_base['loan_status'] = df_base['loan_status'].replace({'Default': 'Charged Off'})
df_base['default_flag'] = (df_base['loan_status'] == 'Charged Off').astype(int)
df_base = df_base.drop(columns='loan_status')

print(df_base["default_flag"].value_counts(dropna=False))
print(df_base['default_flag'].value_counts(normalize=True))

default_flag
0    1041952
1     261686
Name: count, dtype: int64
default_flag
0    0.799265
1    0.200735
Name: proportion, dtype: float64


In [5]:
#Data Cleaning: Removing Columns

#Full empty columns
useless_cols = [
    'id', 'member_id', 'url'
]

#Columns related to hardship program (Post origination)
hardship_cols = [
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd',
    'hardship_loan_status', 'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount'
]
#Columns related to settlement (Post origination/default)
settlement_cols = [
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term'
]

#Post origination Columns
post_origination_cols = [
    "pymnt_plan", "out_prncp", "out_prncp_inv", "total_pymnt",
    "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d"
]

#Fully encoded by sub-grade
deterministic_cols = [
    "grade"
]

joint_cols = [
    'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    'revol_bal_joint','sec_app_earliest_cr_line', 'sec_app_inq_last_6mths',
    'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util',
    'sec_app_open_act_il', 'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths','sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog'
]



drop_cols = useless_cols + hardship_cols + settlement_cols + post_origination_cols + deterministic_cols + joint_cols

df_base = df_base.drop(columns=[c for c in drop_cols if c in df_base.columns])
print("Shape:", df_base.shape)

Shape: (1303638, 91)


In [6]:
#Feature Engineering


#Contains random text
messy_text_cols = [
    "emp_title","desc","title"
]


#Feature Engineering
df_base["emp_title_missing"] = df_base["emp_title"].isna().astype(int)
df_base["desc_missing"] = df_base["desc"].isna().astype(int)
df_base["title_missing"] = df_base["title"].isna().astype(int)


df_base["issue_d"] = pd.to_datetime(df_base["issue_d"], format="%b-%Y")
df_base["issue_year"] = df_base["issue_d"].dt.year

df_base["earliest_cr_line"] = pd.to_datetime(df_base["earliest_cr_line"], format="%b-%Y")

def months_between(later, earlier):
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

df_base["credit_history_months"] = months_between(df_base["issue_d"], df_base["earliest_cr_line"])

df_base["term"] = df_base["term"].str.extract(r"(\d+)").astype("int64")

#dropping 1 singular row where zip code is missing
df_base = df_base.dropna(subset=["zip_code"])
df_base["zip_code"] = (df_base["zip_code"].astype(str).str.extract(r"(\d{3})", expand=False))

df_base["emp_length"] = df_base["emp_length"].fillna("Missing")


#Chronological order
df_base = df_base.sort_values("issue_d").reset_index(drop=True)

drop_cols = messy_text_cols + ["earliest_cr_line"]

df_base = df_base.drop(columns=[c for c in drop_cols if c in df_base.columns])
print("Shape:", df_base.shape)

Shape: (1303637, 92)


In [7]:
#Data Cleaning: Dropping rows before 2016 (Many features not collecting pre 2016)

df_base = df_base[df_base["issue_year"] >= 2016]

# Reset index after filtering
df_base.reset_index(drop=True, inplace=True)

# Inspect new shape
df_base.shape

(480811, 92)

In [9]:
#Feature Categorization, additional feature engineering


ord_cols = [
"sub_grade", "emp_length"
]

ord_cats = [
["A1","A2","A3","A4","A5", 
"B1","B2","B3","B4","B5",
"C1","C2","C3","C4","C5", 
"D1","D2","D3","D4","D5",
"E1","E2","E3","E4","E5", 
"F1","F2","F3","F4","F5",
"G1","G2","G3","G4","G5"],

["Missing", "< 1 year",
"1 year", "2 years",
"3 years", "4 years",
"5 years", "6 years",
"7 years", "8 years",
"9 years", "10+ years"]
]

cat_cols = [
"home_ownership", "verification_status",
"purpose", "zip_code", "addr_state",
"initial_list_status", "application_type",
"disbursement_method"
]

target_col = [
    "default_flag"
]

#Numerical Columns where missingness is structural (Months Since)
num_structural = [
    "mths_since_last_delinq", 
    "mths_since_last_record",  
    "mths_since_last_major_derog",  
    "mths_since_recent_bc_dlq",
    "mths_since_recent_revol_delinq",  
    "mths_since_rcnt_il",
    "mths_since_recent_bc",
    "mths_since_recent_inq",
]

#Create missingness indicator for clarity
for col in num_structural:
    missing_col = f"{col}_missing"
    df_base[missing_col] = df_base[col].isna().astype(int)

exclude = cat_cols + ord_cols + target_col + num_structural + ["issue_d", "issue_year"]

num_rest = [col for col in df_base.columns if col not in exclude]

print(df_base.shape)
print(df_base["default_flag"].value_counts(normalize=True))




(480811, 100)
default_flag
0    0.77118
1    0.22882
Name: proportion, dtype: float64


In [12]:
#Save clean csv file and pkl

data_dir = "../data"

pkl_path = os.path.join(data_dir, "clean_data.pkl")
df_base.to_pickle(pkl_path)
print(f"Saved pickle to: {pkl_path}")

csv_path = os.path.join(data_dir, "clean_data.csv")
df_base.to_csv(csv_path, index=False)
print(f"Saved cleaned CSV dataset to: {csv_path}")



Saved pickle to: ../data/clean_data.pkl
Saved cleaned CSV dataset to: ../data/clean_data.csv
